# Section 1 - Intro to NLP and LLMs

By the end of this workshop you will have built a **RAG system**: a search engine over every NUS module, connected to an LLM that answers questions using what the search finds. This section follows the slides:

1. **Reading text:** preprocessing, tokens and the token budget.
2. **Feature extraction:** keyword search, Bag of Words, TF-IDF and BM25.
3. **Hallucination and RAG:** ask Gemini about NUS modules, check its answers against the real data, and see why RAG is the fix.

## Setup

Run the cells in this part now, while everyone is arriving.

**In VS Code:** click **Select Kernel** (top right of the notebook) → **Python Environments**, and pick the virtual environment where you ran `pip install -r requirements.txt` (run it inside the `week-11-nlp` folder, where `requirements.txt` is). If VS Code asks to install `ipykernel`, click **Install**.

The first cell loads three things through `rag_utils.py`, a small helper file that all three sections share:

- `modules`: every module NUS offers in AY 2026-2027, taken from [NUSMods](https://nusmods.com). Each module is a Python dictionary with its code, title, description, workload and more. The organisers downloaded this in advance and saved it in `data/`, so the workshop does not depend on NUSMods being up.
- `embeddings`: one row of numbers per module. You will meet these in Section 2.
- `model`: `all-MiniLM-L6-v2`, a small model that turns text into numbers. It runs on your own CPU and needs no API key. The first run downloads it (about 90 MB).

In [ ]:
# %pip install -q google-genai sentence-transformers numpy matplotlib
import json
import os
import re
import sys
import textwrap
import urllib.request
import warnings

sys.path.append("..")  # rag_utils.py is one folder up from notebooks/
os.environ["TRANSFORMERS_VERBOSITY"] = os.environ["HF_HUB_VERBOSITY"] = "error"  # hide harmless warnings
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

import rag_utils
from rag_utils import build_chunk, keyword_search, search

modules, embeddings, model = rag_utils.load()
by_code = {module["moduleCode"]: module for module in modules}  # "CS4248" -> its record

print(f"{len(modules)} modules offered in AY {rag_utils.AY}, fetched from NUSMods on {rag_utils.FETCH_DATE}")

In [ ]:
import pandas as pd

# One row per module. workload = weekly hours of [lecture, tutorial, lab, project, preparation]
pd.DataFrame(modules).head(5)

### Connect Gemini

Each of you needs your **own** free API key: the free quota is counted per Google Cloud project, so 40 people sharing one key would run out within minutes.

> **Your Gemini API key (one-time, about 2 minutes)**
> 1. Go to **[aistudio.google.com/apikey](https://aistudio.google.com/apikey)**, sign in with a *personal* Google account, click **Create API key** and copy the key. It's free — **don't** set up billing, because that turns the free tier off for the project.
> 2. Paste the key between the quotes in the cell below, then run it.
>
> The key is saved inside the notebook file, so don't share or commit the notebook with your key in it.

The cell connects `ask_gemini` (from `rag_utils.py`), the one function we use to talk to Gemini. It sends your prompt and nothing else: no extra documents, and no Google Search. If the call fails, for example because the free quota has run out, it prints the reason and returns a saved answer instead of crashing.

You should see `Gemini says: ready`. If you see `Gemini call failed` instead, the message tells you why. The most common one is `API key not valid`: copy the key again and re-run the cell. If it still fails, raise your hand. You can follow the Gemini demos on the main screen in the meantime.

In [ ]:
from rag_utils import ask_gemini

GEMINI_API_KEY = ""  # vscode will prompt you to enter this in a secure way, or you can hardcode it here if you prefer

rag_utils.connect_gemini(GEMINI_API_KEY)
print("Gemini says:", ask_gemini("Reply with exactly: ready"))

## 1.1 Reading text

How does a computer read text? Computers only understand numbers, so we have to turn words into numbers. That is hard, because text is **messy** (punctuation, capital letters, codes like `CS3244`) and **ambiguous** (one word can mean different things, and different words can mean the same thing).

NLP deals with this through a pipeline. This section covers the first six steps, and Section 2 the last two:

**Raw text → Cleaning → Tokenization → Normalization → Chunking → Feature extraction** → Embedding → Retrieval

Our example all the way through is one real NUSMods description, the first sentence of **CS3244 Machine Learning**:

> This course introduces basic concepts and algorithms in machine learning and neural networks.

## 1.2 Preprocessing

Before a computer can count words, we clean the text up in three stages:

- **Cleaning:** remove punctuation and symbols.
- **Tokenization:** split into words.
- **Normalization:** make different forms of the same word look alike. That covers lowercasing, stopword removal (drop common words such as "this", "and", "in"), stemming (chop off word endings: `machine` → `machin`) and lemmatization (back to the dictionary form: `networks` → `network`).

These steps help methods that count words, such as BM25 in 1.4. MiniLM, the model we use in Section 2, does not need most of them: it lowercases by itself and has its own tokenizer.

## 1.3 Tokenization

A **token** is a fundamental chunk of text that a language model processes, and **tokenization** is splitting text into tokens. Common words are usually 1 token, while rare words and codes get split into smaller pieces, e.g. `CS3244` → `CS | 324 | 4`.

A model can only read a limited number of tokens at once: its **token budget**. Anything past the budget gets cut off, and the model never sees it. **Chunking** cuts long text into pieces that fit the budget. You will choose chunks yourself in Section 2.

Here is MiniLM's tokenizer reading the CS3244 sentence. Change `budget` and re-run to see where the sentence gets cut.

In [ ]:
sentence = by_code["CS3244"]["description"].split(". ")[0] + "."
tokens = model.tokenizer.tokenize(sentence)
print(f"{len(tokens)} tokens:", tokens)

# Try it: change the budget, then re-run
budget = 10

print()
print(f"Read ({budget} tokens):", model.tokenizer.convert_tokens_to_string(tokens[:budget]))
print(f"Cut off ({len(tokens[budget:])} tokens):", model.tokenizer.convert_tokens_to_string(tokens[budget:]))
print(f"MiniLM's real budget: {model.max_seq_length} tokens")

With a budget of 10, the model never finds out that CS3244 covers neural networks. Real budgets are bigger, but so is real text.

**YOUR TURN:** every model comes with its own tokenizer. Alter the words below and re-run to see how MiniLM splits them, then paste the same words into ChatGPT's tokenizer at [tiktokenizer.vercel.app](https://tiktokenizer.vercel.app/) and compare. Try other module codes, your name, Singlish, or another language.

**Think about it:** how does the model decide where to split?

In [ ]:
# YOUR TURN: alter the words, then re-run
my_words = [
    "learning", "GPU", "CS3244", "NUSMods",      # from the slides
    "regularities", "self-learning", "CS2040S",  # from the CS3244 description, and one of its prerequisites
    "eﬃciency", "汉语拼音方案",                    # oddities from the FIN3702A and CL3211 descriptions
]

for word in my_words:
    print(f"{word:>15} -> {model.tokenizer.tokenize(word)}")

In [ ]:
# The model only sees ID numbers, and decode turns the numbers back into text
# Try it: change the text, then re-run
text = "CS3244 introduces neural networks"

ids = model.tokenizer.encode(text, add_special_tokens=False)
print("Token IDs:", ids)
print("Tokens:   ", model.tokenizer.convert_ids_to_tokens(ids))
print("Decoded:  ", model.tokenizer.decode(ids))

### Subword encoding

Storing every whole word would take too many pieces, and still fail on a new word such as "NUSMods". Single letters can spell anything, but make text far too long. **Subwords** sit in between: break words into smaller pieces that can be reused.

- Common words stay whole: `course` → `[course]`.
- Rare words are built from known pieces: `CS3244` → `[CS | 324 | 4]`.

So the model can read new words with a fixed list of pieces. Two common ways to build that list:

- **Byte-Pair Encoding (BPE):** start from single letters, and keep gluing together the two pieces that appear side by side most often. On `learning`, `training` and `mining`, round 1 glues `i + n → in`, and round 2 glues `in + g → ing`. New words reuse the pieces: `coding` → `c | o | d | ing`. ChatGPT's tokenizer is a BPE tokenizer.
- **WordPiece:** BPE's cousin. Same idea, but BPE glues the most common pair, while WordPiece glues the pair that sticks together most (pieces that rarely show up apart). Pieces inside a word start with `##`: `GPU` → `[gp | ##u]`.

MiniLM uses WordPiece, which is where the `##` in your output comes from. It also lowercases everything first, which is why decoding gives back `cs3244`, not `CS3244`. `[UNK]` means **unknown**: the character is not in MiniLM's vocabulary at all. `eﬃciency` looks normal, but its "ﬃ" is one special character (probably copied from a PDF), so MiniLM cannot read the word. That is messy text in action.

## 1.4 Feature extraction

### The search process

RAG has to find the right modules first. The simplest way is **keyword search**: return every module whose title contains your exact words, like the search bar on NUSMods. It is exact, but it ignores similar modules we might want: "machine learning" finds CS3244 Machine Learning, but not AIS5103 Foundations of Deep Learning.

The idea: look at all the words in the description, not just the title. `keyword_search` (from `rag_utils.py`) does that, like Ctrl+F on every description.

**YOUR TURN:** change the query and compare the two searches.

In [ ]:
def title_search(query):
    """Modules whose title contains the query, word for word."""
    return [module for module in modules if query.lower() in module["title"].lower()]


# YOUR TURN: change the query, then re-run
query = "machine learning"

for where, matches in [("titles", title_search(query)), ("descriptions", keyword_search(query))]:
    print(f'"{query}" in {where}: {len(matches)} modules')
    for match in matches[:5]:
        print(f"    {match['moduleCode']:<9} {match['title']}")

The descriptions give many more matches, AIS5103 among them, but two problems remain. The matches come back in alphabetical order, not by relevance. And both searches compare characters: try `"how computers understand language"`, and neither finds anything, because no title or description contains that exact phrase.

### Bag of Words, TF-IDF and BM25

To rank modules, we turn each description into numbers (**feature extraction**) and give every module a score:

- **Bag of Words (BoW):** count each word in a description, so the text becomes a row of numbers. The score is the count of shared words. Problems: every word counts the same, so "course", which is in almost every description, still earns points; and word order is lost ("students teach tutors" = "tutors teach students").
- **TF-IDF:** TF (term frequency) is how often the word appears in this description, divided by its length. IDF (inverse document frequency) is how rare the word is across all descriptions: $\log_{10}(N \div \text{descriptions that have the word})$. TF × IDF is high for a word that is frequent here and rare elsewhere.
- **BM25 (Best Matching 25):** TF-IDF with two fixes. **Saturation:** the 10th "learning" adds much less than the 1st (`k1 = 1.2`). **Length:** long descriptions don't win just by having more words (`b = 0.75`).

The next cell scores every NUSMods description all three ways, with the same formulas as the tables on the slides.

In [ ]:
from collections import Counter


def split_words(text):
    """Lowercase the text and split it into words."""
    return re.findall(r"[a-z0-9]+", text.lower())


bags = [Counter(split_words(module["description"])) for module in modules]  # one Bag of Words per module
lengths = np.array([sum(bag.values()) for bag in bags])                       # words in each description
has_word = Counter(word for bag in bags for word in bag)                      # descriptions that have each word


def word_scores(query, method="bm25", k1=1.2, b=0.75):
    """Score every description against the query with "bow", "tfidf" or "bm25"."""
    scores = np.zeros(len(modules))
    for word in split_words(query):
        count = np.array([bag[word] for bag in bags])
        idf = np.log10(len(modules) / has_word[word]) if has_word[word] else 0.0
        if method == "bow":
            scores += count
        elif method == "tfidf":
            scores += count / lengths * idf
        elif method == "bm25":
            scores += idf * count * (k1 + 1) / (count + k1 * (1 - b + b * lengths / lengths.mean()))
    return scores


def word_search(query, method="bm25", k=5):
    """Return the top k (module, score) pairs."""
    scores = word_scores(query, method)
    top = np.argsort(-scores)[:k]
    return [(modules[i], float(scores[i])) for i in top]

In [ ]:
# YOUR TURN: change the query, then re-run
query = "machine learning course"

for method in ["bow", "tfidf", "bm25"]:
    print(method)
    for module, score in word_search(query, method):
        print(f"  {score:6.2f}  {module['moduleCode']:<9} {module['title']:<45} ({len(split_words(module['description']))} words)")

**How to read it:** look at the word counts in brackets. A typical description has about 90 words.

- **BoW** puts EBA5012 first, a 243-word description: long text repeats the query words more often.
- **TF-IDF** divides by the length, so short descriptions such as SA4110 (17 words) jump to the top.
- **BM25** caps what repeated words add and compares each length with the average, so long and short descriptions compete on fairer terms.

"course" adds almost nothing to TF-IDF and BM25: nearly every description has it, so its IDF is close to 0. Try your own query, such as `"how computers understand language"`. Do the three methods agree?

## 1.5 Hallucination

When we use an LLM to look something up, it might **not know the updated data**, or **invent fluent but untrue facts**. Two reasons:

- **Knowledge cutoff:** it only knows what it was trained on, up to a certain date.
- **Next-token prediction:** it writes one token at a time, picking what sounds likely, not what is true.

Confident answers that are wrong are called **hallucinations**. We will ask Gemini about NUS modules, giving it **no extra information**, and then **check its answer** against the real NUSMods data.

`verify` does the checking. It finds every module code in an answer, such as `CS3244`, and looks it up in this year's module list.

In [ ]:
MODULE_CODE = re.compile(r"\b[A-Z]{2,4}\d{4}[A-Z]{0,2}\b")  # e.g. CS3244, CS2040S


def verify(answer):
    """Look up every module code in the answer in the real NUSMods data."""
    codes = list(dict.fromkeys(MODULE_CODE.findall(answer)))  # unique, in order
    for code in codes:
        real = by_code.get(code)
        if real:
            print(f"  ✅ {code:<9} real title: {real['title']}")
        else:
            print(f"  ❌ {code:<9} not offered in AY {rag_utils.AY}, or does not exist")
        said = next(line for line in answer.splitlines() if code in line)  # where Gemini mentioned it
        print(f"{'':15}Gemini wrote: {textwrap.shorten(re.sub(r'[*_#`|>]', '', said), 90)}")
    print()
    print(f"{sum(code in by_code for code in codes)} of {len(codes)} module codes are real.")

In [ ]:
question = "Which NUS modules cover attention mechanisms? List the module codes and titles."

answer = ask_gemini(question)
display(Markdown(answer))
print("-" * 60)
verify(answer)

**How to read it:**

- ❌ means the code is not in this year's NUSMods data. Either the module does not exist, or it is not running this year.
- ✅ with a different title means the code is real, but Gemini attached the wrong module to it, or used an old name.
- ✅ with a matching title means Gemini got that one right. You only know because you checked.

### Without and with retrieval

The surest way to catch an LLM inventing things is to ask about something that does not exist. `CS3246` looks like a real CS module, right next to CS3244, but NUS has never offered it. The cell below first checks NUSMods, then asks Gemini twice: once on its own, and once with what NUSMods has pasted into the prompt.

In [ ]:
module_code = "CS3246"  # made up: NUS has never offered a module with this code
module_question = (f"What is the NUS module {module_code} about? "
                   "Give its title, a short description and its weekly workload in hours.")

print(f"Is {module_code} in NUSMods AY {rag_utils.AY}?", module_code in by_code)
print()
print("WITHOUT RETRIEVAL:")
print(ask_gemini(module_question))
print()

# What NUSMods has: the module's entry, or a note that there is none
if module_code in by_code:
    entry = build_chunk(by_code[module_code])
else:
    entry = f"NUSMods has no module {module_code} in AY {rag_utils.AY}."
print("WITH WHAT NUSMODS HAS:")
print(ask_gemini(f"Answer using only this information from NUSMods.\n\n{entry}\n\n{module_question}"))

Without retrieval, Gemini may describe a module that does not exist, with a title, topics and workload hours that all sound reasonable: that is exactly what next-token prediction produces. With what NUSMods has in front of it, Gemini can tell you the module does not exist, and you know where that answer came from: NUSMods. Hold on to that idea, because it is the core of 1.6. Change `module_code` to a module you have taken, such as `CS2040S`, to see both answers for a real module.

**YOUR TURN:** ask Gemini your own question about NUS modules, then check it. Ask for module codes, so that `verify` has something to check. Some things to try:

- A topic from the CS3244 description: *"Which NUS modules teach neural networks?"*
- What comes next: *"Which NUS modules should I take after CS3244?"*
- Something that changes every year, to test the knowledge cutoff: *"Which modules are precluded for CS3244?"*

In [ ]:
# YOUR TURN: ask your own question, then re-run
my_question = "Which NUS modules should I take after CS3244 Machine Learning? List the module codes and titles."

my_answer = ask_gemini(my_question)
display(Markdown(my_answer))
print("-" * 60)
verify(my_answer)

## 1.6 The concept of RAG

**Retrieval-Augmented Generation (RAG)** is a framework that improves an LLM by connecting it to an external knowledge source. Ours is NUSMods.

Why not paste all of NUSMods into the prompt, the way we pasted the CS3244 entry? Because of the **token budget**: there are too many descriptions to fit into a single prompt. The cell below counts tokens with MiniLM's tokenizer, so treat the numbers as estimates.

In [ ]:
chunks = [build_chunk(m) for m in modules]
chunk_lengths = [len(ids) for ids in model.tokenizer(chunks, verbose=False)["input_ids"]]

print(f"CS3244 alone: about {chunk_lengths[modules.index(by_code['CS3244'])]} tokens")
print(f"All {len(modules)} modules: about {sum(chunk_lengths):,} tokens, in every single prompt")
print(f"5 relevant modules: about {sum(chunk_lengths[:5]):,} tokens")

try:
    gemini_limit = rag_utils.gemini_client().models.get(model=rag_utils.GEMINI_MODEL).input_token_limit
    print(f"{rag_utils.GEMINI_MODEL} reads at most {gemini_limit:,} tokens per prompt")
except Exception:
    pass

All of NUSMods would fill most of the budget, if it fits at all, and every single question would pay for it again, when an answer needs only a handful of modules. **So retrieve only the most relevant modules, and let the LLM answer from those.** A RAG system does that in seven steps:

| Step | What happens | Where |
|---|---|---|
| 1. Encode, 2. Index | the embedding model turns every module into a vector, and we store the vectors | Section 2 |
| 3. Encode, 4. Similarity search | the query becomes a vector too, and we find the module vectors closest to it | Section 2 |
| 5. Similar documents | the top few modules come back as the retrieved context | Section 2 |
| 6. Prompt | the question and the retrieved modules go to the LLM together | Section 3 |
| 7. Final response | the LLM answers from them, citing its sources | Section 3 |

## Recap

So far: text → tokens → word counts (Bag of Words, TF-IDF, BM25) → find the best-matching modules → give them to the LLM (RAG).

But keyword methods only match the exact same words. Search for "artificial intelligence": the CS3244 description talks about "machine learning and neural networks", so the two share 0 words, and BM25 scores CS3244 at 0, even though it is clearly an AI module.

In [ ]:
cs3244 = modules.index(by_code["CS3244"])

print('BM25 score of CS3244 for "artificial intelligence":', word_scores("artificial intelligence", "bm25")[cs3244])
print('Ctrl+F matches for "how computers understand language":', len(keyword_search("how computers understand language")))

Word counts still do not capture meaning, which is why we need **embeddings**. That is where Section 2 starts.

### If you're stuck

Run the cell below. It reloads everything Section 2 needs.

In [ ]:
# Reloads the data, the embedding model and BM25 if anything above failed
import re
import sys
from collections import Counter

sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np

import rag_utils
from rag_utils import ask_gemini, build_chunk, keyword_search, search

modules, embeddings, model = rag_utils.load()
by_code = {module["moduleCode"]: module for module in modules}


# From Section 1.4: BM25
def split_words(text):
    """Lowercase the text and split it into words."""
    return re.findall(r"[a-z0-9]+", text.lower())


bags = [Counter(split_words(module["description"])) for module in modules]
lengths = np.array([sum(bag.values()) for bag in bags])
has_word = Counter(word for bag in bags for word in bag)


def word_scores(query, method="bm25", k1=1.2, b=0.75):
    """Score every description against the query with "bow", "tfidf" or "bm25"."""
    scores = np.zeros(len(modules))
    for word in split_words(query):
        count = np.array([bag[word] for bag in bags])
        idf = np.log10(len(modules) / has_word[word]) if has_word[word] else 0.0
        if method == "bow":
            scores += count
        elif method == "tfidf":
            scores += count / lengths * idf
        elif method == "bm25":
            scores += idf * count * (k1 + 1) / (count + k1 * (1 - b + b * lengths / lengths.mean()))
    return scores


print(len(modules), "modules, embeddings", embeddings.shape)